In [ ]:
import asf_search as asf
import pandas as pd

aoi =  "POLYGON((86.2958 27.9437,86.2942 28.0251,86.133 28.0214,86.1284 27.8743,86.3035 27.8761,86.2958 27.9437))"
# kml file bounded polygon, constant for now

start_date = "2024-01-01T00:00:00Z"
end_date = "2025-05-01T00:00:00Z"

print("Searching ASF for Sentinel-1 SLC time-series stack...")
results = asf.geo_search(
    platform=asf.PLATFORM.SENTINEL1,
    processingLevel=asf.PRODUCT_TYPE.SLC,
    beamMode=asf.BEAMMODE.IW,
    flightDirection=asf.FLIGHT_DIRECTION.DESCENDING,
    intersectsWith=aoi,
    start=start_date,
    end=end_date,
)

print(f"Found {len(results)} scenes.")

# Convert metadata to a DataFrame

metadata = [r.properties for r in results]
df_og = pd.DataFrame(metadata)

df = df_og.copy()

# Drop redundant or constant columns to reduce noise
columns_to_drop = [
    'flightDirection',
    'processingLevel',
    'url',
    'sensor',
    'beamModeType',
    's3Urls',
    'browse',
    'md5sum',
    'bytes',
    'pgeVersion',
    'fileName',
    'fileID'
]
df = df.drop(columns=columns_to_drop, errors='ignore')
# for api pulls, ignore is better
# if to_drop columns that dont exist, we go head and IGNORE
df.sample(3)

In [ ]:
import zipfile
import glob
import os

# Target Directory & Extraction Setup
download_dir = os.path.expanduser(r"C:\Users\digit\Desktop\test slc")
extract_dir = os.path.join(download_dir, "extracted_tifs")
os.makedirs(extract_dir, exist_ok=True)

print(f"[*] Scanning {download_dir} for HyP3 zip files...\n")
zip_files = glob.glob(os.path.join(download_dir, "S1AA_*_INT80_*.zip"))
total_zips = len(zip_files)

if total_zips == 0:
    print("[!] FATAL: No HyP3 zip files found in the specified directory.")
else:
    print(f"[*] FOUND {total_zips} ZIP FILES. Commencing Batch Extraction...\n")

    extracted_paths = []

    # Batch Loop over all ZIPs
    for i, z_path in enumerate(zip_files, 1):
        filename = os.path.basename(z_path)
        print(f"[{i}/{total_zips}] TARGET: {filename}")

        try:
            with zipfile.ZipFile(z_path) as z:
                zip_contents = z.namelist()
                coh_files = [f for f in zip_contents if f.endswith('corr.tif') or f.endswith('coh.tif')]

                if not coh_files:
                    print(f"    [!] No coherence file found. Skipping...")
                    continue

                coh_file = coh_files[0]
                out_name = os.path.basename(coh_file)
                current_extracted_path = os.path.join(extract_dir, out_name)

                # Skip if already extracted
                if os.path.exists(current_extracted_path):
                    print(f"    [=] Already extracted: {out_name}")
                    extracted_paths.append(current_extracted_path)
                    continue

                print(f"    [*] Extracting: {out_name} ...")
                with z.open(coh_file) as source, open(current_extracted_path, "wb") as target:
                    target.write(source.read())

                extracted_paths.append(current_extracted_path)
                print(f"    [+] Done.")
        except zipfile.BadZipFile:
            print(f"    [!] FATAL: {filename} is corrupted or incompletely downloaded.")

    print(f"\n[+] BATCH EXTRACTION COMPLETE. {len(extracted_paths)} files ready on disk.")

    # Update variable for the next cell to plot the LAST extracted file if desired
    if extracted_paths:
        extracted_path = extracted_paths[0]

In [ ]:
import rasterio
import matplotlib.pyplot as plt

print(f"[*] LOCATING EXTRACTED FILE on disk: {extracted_path}")

if not os.path.exists(extracted_path):
    print("[!] FATAL: File does not exist on disk.")
else:
    print("[*] Opening raster with rasterio...")
    # Read the physically extracted file into memory
    with rasterio.open(extracted_path) as src:
        print(f"[*] MATRIX METADATA: {src.width} columns x {src.height} rows")
        print(f"[*] COORDINATE SYSTEM: {src.crs}")
        print("[*] Reading 2D float array into RAM...")
        coherence_data = src.read(1)

    print("[*] Data loaded successfully. Triggering Matplotlib render...")

    # Plot the coherence
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(coherence_data, cmap='gray', vmin=0, vmax=1)

    # UI adjustments
    plt.colorbar(im, label="Coherence (0 to 1)")
    ax.set_title("Coherence Map (1=Stable, 0=Noise)")
    ax.axis('off')

    print("[+] RENDER COMPLETE.")
    plt.show()